# Pretraining an embedding, and the geometry it learns

Continuous bag-of-words on unlabelled text, then use of the result — and a look at the space itself, which is where the interest is.

**Runs on:** CPU — about 10 minutes &nbsp;·&nbsp; **Slides:** [Chapter 14 — Text Classification](../../../course-web-slides/ch14/index.html) &nbsp;·&nbsp; **Section:** 04 — Pretraining word embeddings

---

## The idea

Chapter 8's argument in a new modality. **Labelled data is scarce; unlabelled text is not.** Train an embedding on a task that needs no labels — predict a word from its neighbours — and reuse the geometry.

In [ ]:
import numpy as np
import keras
from keras import layers

# A corpus. Substitute anything larger you have; more text is strictly better.
from keras.datasets import imdb
word_index = imdb.get_word_index()
index_word = {v + 3: k for k, v in word_index.items()}
index_word.update({0: "<PAD>", 1: "<START>", 2: "<UNK>"})

(train_data, _), _ = imdb.load_data(num_words=10000)
print(f"{len(train_data):,} documents, "
      f"{sum(len(d) for d in train_data):,} tokens")

## Building (context, target) pairs

In [ ]:
WINDOW = 2
VOCAB = 10000

def cbow_pairs(sequences, window=WINDOW, limit=400_000):
    contexts, targets = [], []
    for seq in sequences:
        for i, target in enumerate(seq):
            lo, hi = max(0, i - window), min(len(seq), i + window + 1)
            ctx = [seq[j] for j in range(lo, hi) if j != i]
            if len(ctx) != 2 * window:
                continue
            contexts.append(ctx)
            targets.append(target)
            if len(targets) >= limit:
                return np.array(contexts), np.array(targets)
    return np.array(contexts), np.array(targets)

X, y = cbow_pairs(train_data)
print("contexts:", X.shape, " targets:", y.shape)
print("\nexample:")
print("  context:", [index_word.get(i, '?') for i in X[100]])
print("  target: ", index_word.get(y[100], '?'))

**No labels anywhere.** The supervision comes from the text's own structure, which is the definition of self-supervised learning and the same principle behind every model in chapters 15 to 17.

## The CBOW model

In [ ]:
EMBED = 128

inputs = keras.Input(shape=(2 * WINDOW,), dtype="int32")
emb = layers.Embedding(VOCAB, EMBED, name="embedding")(inputs)
x = layers.GlobalAveragePooling1D()(emb)     # average the context vectors
outputs = layers.Dense(VOCAB, activation="softmax")(x)
cbow = keras.Model(inputs, outputs)

cbow.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
             metrics=["accuracy"])
cbow.summary()

Average the context embeddings, predict the missing word. The output layer is 10,000-wide and holds most of the parameters — **and it is thrown away**. The `Embedding` table is the artifact we are after.

## Training

In [ ]:
cbow.fit(X, y, epochs=5, batch_size=512, validation_split=0.05,
         verbose=2)

> **Note** — Accuracy will be low — predicting one word out of 10,000 from four neighbours is genuinely hard. **The accuracy is not the objective**; the embedding table is.

## Looking at the space

In [ ]:
W = cbow.get_layer("embedding").get_weights()[0]
norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-9
Wn = W / norms

def nearest(word, k=8):
    idx = word_index.get(word)
    if idx is None or idx + 3 >= VOCAB:
        return f"{word!r} not in the vocabulary"
    i = idx + 3
    sims = Wn @ Wn[i]
    top = np.argsort(sims)[::-1][1:k+1]
    return [(index_word.get(j, "?"), round(float(sims[j]), 3)) for j in top]

for w in ["good", "terrible", "film", "actor", "three"]:
    print(f"{w:10s} ->", nearest(w, 6))

The neighbours should be **semantically related, not orthographically similar**. Expect *good* near *great*, *decent*, *fine*; *three* near other numbers.

Nothing told the model that numbers form a category. It fell out of the fact that numbers appear in similar contexts — which is the whole distributional hypothesis, demonstrated.

## Projecting it

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

words = ["good", "great", "excellent", "wonderful", "best",
         "bad", "terrible", "awful", "worst", "poor",
         "one", "two", "three", "four", "five",
         "he", "she", "they", "him", "her",
         "movie", "film", "story", "plot", "script"]
idxs = [word_index[w] + 3 for w in words if w in word_index
        and word_index[w] + 3 < VOCAB]
labels = [index_word[i] for i in idxs]

proj = TSNE(n_components=2, perplexity=6, init="pca",
            random_state=0).fit_transform(Wn[idxs])

plt.figure(figsize=(9, 7))
plt.scatter(proj[:, 0], proj[:, 1], s=30, c="#00539f")
for (x0, y0), lab in zip(proj, labels):
    plt.annotate(lab, (x0, y0), fontsize=10,
                 xytext=(4, 4), textcoords="offset points")
plt.title("A learned word geometry"); plt.xticks([]); plt.yticks([])
plt.show()

Positive adjectives together, negative adjectives together, numbers together, pronouns together. **This geometry is what chapter 15 means by an embedding space**, and what its attention mechanism repeatedly refines.

## Using it downstream

In [ ]:
# Freeze the pretrained table and train only the classifier on top.
pretrained = layers.Embedding(VOCAB, EMBED, trainable=False,
                              name="pretrained_embedding")
pretrained.build((None,))
pretrained.set_weights([W])

inputs = keras.Input(shape=(None,), dtype="int32")
x = pretrained(inputs)
x = layers.Bidirectional(layers.LSTM(32))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
clf = keras.Model(inputs, outputs)
clf.compile(optimizer="rmsprop", loss="binary_crossentropy",
            metrics=["accuracy"])
print(f"trainable parameters: {sum(np.prod(w.shape) for w in clf.trainable_weights):,}")
print(f"frozen (pretrained):  {np.prod(W.shape):,}")

The chapter-8 procedure, exactly: **freeze, train the head, consider unfreezing later at a lower learning rate.**

Pretrained embeddings help most when labelled data is scarce. On the full IMDB training set they help little — 20,000 labelled reviews is enough to learn a task-specific geometry. Try it with 1,000 and the gap opens.

## What chapter 15 changes about all this

One vector per word, fixed. *Bank* gets a single embedding whether it is a river bank or a savings bank.

**Chapter 15's attention mechanism produces a different vector for each occurrence**, conditioned on its neighbours. That is the single largest step between this notebook and a modern language model — and the geometry you plotted above is the thing it refines, layer by layer.

---

## What to take away

- CBOW learns an embedding from unlabelled text — self-supervision, as in chapters 15 to 17.
- The output layer holds most of the parameters and is discarded; the table is the artifact.
- The learned geometry groups words by **context**, which is the distributional hypothesis working.
- One vector per word is the limitation attention removes.